In [16]:
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
from fonction import *
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

In [5]:
gdf = gpd.read_file('data/processed/jointure_meteo_swimonthly_full.gpkg')

In [6]:
display(gdf)

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,PRENEI,PRELIQ,T,FF,Q,...,HTEURNEIGE,HTEURNEIGE6,HTEURNEIGEX,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,geometry
0,2,641374,7106309,1960-01-01,0.863,4.9,51.5,4.829032,6.822581,4.584839,...,0.001710,0.001839,0.024,0.096774,4.0,0.287032,0.002484,-7.2,11.6,POINT (588001.454 2673000.704)
1,7119,635809,6442801,1960-01-01,1.081,20.2,96.4,1.990323,2.061290,3.926677,...,0.030935,0.031323,0.174,0.177419,38.0,0.298742,0.012194,-13.1,11.7,POINT (587999.304 2008998.388)
2,7118,627817,6442868,1960-01-01,1.100,22.7,119.8,2.783871,1.996774,4.188290,...,0.032935,0.033581,0.180,0.203226,50.2,0.264548,0.012129,-12.8,12.2,POINT (579999.298 2008998.399)
3,7117,619825,6442935,1960-01-01,1.095,21.8,115.4,3.277419,2.083871,4.315355,...,0.030839,0.031226,0.170,0.180645,45.7,0.284097,0.010065,-12.2,13.0,POINT (571999.291 2008998.458)
4,7116,611833,6443002,1960-01-01,1.086,23.8,108.4,3.490323,2.174194,4.422677,...,0.032871,0.033323,0.179,0.183871,41.7,0.293774,0.008742,-11.5,13.3,POINT (563999.284 2008998.565)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7005175,3831,805955,6713138,2024-12-01,0.927,0.9,68.4,4.016129,2.809677,4.880258,...,0.000000,0.000000,0.000,0.000000,0.9,0.344613,0.000097,-3.9,12.8,POINT (756000.273 2280998.1)
7005176,3832,813948,6713070,2024-12-01,0.908,1.5,71.1,2.861290,3.348387,4.495484,...,0.000000,0.000000,0.001,0.000000,1.8,0.324968,0.000613,-5.0,11.4,POINT (763999.644 2280998.186)
7005177,3833,821942,6713002,2024-12-01,0.923,3.2,71.2,2.835484,3.409677,4.498032,...,0.000000,0.000000,0.001,0.000000,3.9,0.321000,0.000516,-4.8,11.5,POINT (772000.016 2280998.33)
7005178,3827,773980,6713410,2024-12-01,0.973,0.0,64.9,4.154839,2.825806,4.791226,...,0.000000,0.000000,0.000,0.000000,0.0,0.375032,0.000774,-2.5,13.3,POINT (723999.802 2280998.23)


In [7]:
y = gdf["SWI_UNIF_MENS"]

meteo_vars = [
    "T", "Q", "FF",
    "PRENEI", "PRELIQ",
    "SSI", "DLI",
    "HTEURNEIGE", "SNOW_FRAC",
    "ECOULEMENT",
    "WG_RACINE", "WGI_RACINE",
    "TINF_H", "TSUP_H"
]

In [8]:
gdf = gdf.copy()
gdf["month"] = gdf["DATE"].dt.month
gdf["sin_month"] = np.sin(2 * np.pi * gdf["month"] / 12)
gdf["cos_month"] = np.cos(2 * np.pi * gdf["month"] / 12)

gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y

In [9]:
X_cols = (
    meteo_vars +
    ["sin_month", "cos_month", "x", "y"]
)

X = gdf[X_cols]
y = gdf["SWI_UNIF_MENS"]

In [10]:
X

,T,Q,FF,PRENEI,PRELIQ,SSI,DLI,HTEURNEIGE,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,sin_month,cos_month,x,y
0,4.829032,4.584839,6.822581,4.9,51.5,5668.6,85952.0,0.001710,0.096774,4.0,0.287032,0.002484,-7.2,11.6,5.000000e-01,0.866025,5.880015e+05,2.673001e+06
1,1.990323,3.926677,2.061290,20.2,96.4,14448.3,73603.7,0.030935,0.177419,38.0,0.298742,0.012194,-13.1,11.7,5.000000e-01,0.866025,5.879993e+05,2.008998e+06
2,2.783871,4.188290,1.996774,22.7,119.8,15546.2,71915.4,0.032935,0.203226,50.2,0.264548,0.012129,-12.8,12.2,5.000000e-01,0.866025,5.799993e+05,2.008998e+06
3,3.277419,4.315355,2.083871,21.8,115.4,15365.5,72581.4,0.030839,0.180645,45.7,0.284097,0.010065,-12.2,13.0,5.000000e-01,0.866025,5.719993e+05,2.008998e+06
4,3.490323,4.422677,2.174194,23.8,108.4,15037.1,73627.8,0.032871,0.183871,41.7,0.293774,0.008742,-11.5,13.3,5.000000e-01,0.866025,5.639993e+05,2.008999e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7005175,4.016129,4.880258,2.809677,0.9,68.4,10204.5,84831.5,0.000000,0.000000,0.9,0.344613,0.000097,-3.9,12.8,-2.449294e-16,1.000000,7.560003e+05,2.280998e+06
7005176,2.861290,4.495484,3.348387,1.5,71.1,9272.6,84829.3,0.000000,0.000000,1.8,0.324968,0.000613,-5.0,11.4,-2.449294e-16,1.000000,7.639996e+05,2.280998e+06
7005177,2.835484,4.498032,3.409677,3.2,71.2,9420.0,83560.9,0.000000,0.000000,3.9,0.321000,0.000516,-4.8,11.5,-2.449294e-16,1.000000,7.720000e+05,2.280998e+06
7005178,4.154839,4.791226,2.825806,0.0,64.9,8200.5,81367.8,0.000000,0.000000,0.0,0.375032,0.000774,-2.5,13.3,-2.449294e-16,1.000000,7.239998e+05,2.280998e+06


In [11]:
train = gdf[gdf["DATE"] < "2018-01-01"]
test  = gdf[gdf["DATE"] >= "2018-01-01"]

X_train = train[X_cols]
y_train = train["SWI_UNIF_MENS"]

X_test = test[X_cols]
y_test = test["SWI_UNIF_MENS"]

In [ ]:
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("reg", Ridge(alpha=1.0))
    ]
)

In [28]:
from sklearn import metrics

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Print the complete summary of the model's performance
print("Model coefficients:", model.named_steps['reg'].coef_)
print("Model intercept:", model.named_steps['reg'].intercept_)
print("R² (train):", model.score(X_train, y_train))
print("R² (test):", model.score(X_test, y_test))
print("RMSE (train):", np.sqrt(metrics.mean_squared_error(y_train, model.predict(X_train))))
print("RMSE (test):", np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

Model coefficients: [-0.20427573  0.11811369 -0.02023296  0.01094149  0.07143714 -0.00892247
 -0.02959015 -0.00955322 -0.02065648  0.02161763  0.094615    0.01332031
  0.0254982  -0.04263763  0.13538385 -0.05074013 -0.0071835   0.00355194]
Model intercept: 0.61735012628832
R² (train): 0.7942477544506177
R² (test): 0.8249452658441506
RMSE (train): 0.14758000081949657
RMSE (test): 0.14276823744045217


In [26]:
y_test

6250776    1.083
6250777    1.083
6250778    1.090
6250779    1.080
6250780    1.174
           ...  
7005175    0.927
7005176    0.908
7005177    0.923
7005178    0.973
7005179    0.177
Name: SWI_UNIF_MENS, Length: 754404, dtype: float64

In [24]:
y_pred

array([1.07484535, 1.07106793, 1.08914599, ..., 0.819295  , 0.90421076,
       0.40410732], shape=(754404,))